> Notebook-friendly copy of `appendix/01-terminal-commands.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

# Terminal Commands

The terminal is a text interface for talking to your computer directly, without a mouse. Every command below has an exact unix form (macOS and Linux) and an exact PowerShell form (Windows); do not try to memorise either column — bookmark this page and copy what you need.

**🎯 Learning objectives**

- Open a terminal on macOS, Linux, and Windows.
- Move around the filesystem, and read, copy, move, and delete files from the command line.
- Build os-independent paths in python with pathlib.
- Run a python script and manage a virtual environment from the shell.

## Opening a terminal

- **macOS** — in Applications -> Utilities (or search "Terminal"); the default shell is zsh.
- **Linux** — usually Ctrl+Alt+T, or a "Terminal" application in your desktop's menu; the default shell is usually bash.
- **Windows** — search "PowerShell" (built in), or install the newer Windows Terminal app from the Microsoft Store. For a real unix-like shell, see the WSL box below.

## Navigation

Three commands cover the basics: print the working directory, list its contents, change directory.

```bash
# macOS (zsh) and Linux (bash)
pwd                   # print working directory
ls -la                # list all files, long format
cd ../data            # up one level, then into data/
```

```powershell
# Windows (PowerShell)
Get-Location          # print working directory (alias: pwd)
Get-ChildItem         # list files (aliases: ls, dir)
Set-Location ..\data  # change directory (alias: cd ..\data)
```

The most visible difference is the path separator: unix uses `/`, Windows uses `\`. Hard-coding either makes code non-portable — see the pathlib section below.

## Files and directories

```bash
# macOS / Linux
mkdir results                     # create a directory
touch notes.txt                   # create an empty file
cp notes.txt notes_backup.txt     # copy
mv notes_backup.txt archive/      # move (or rename)
rm notes_backup.txt               # delete a file
cat notes.txt                     # print a file's contents
```

```powershell
# Windows (PowerShell)
New-Item -ItemType Directory results   # create a directory (alias: mkdir)
New-Item notes.txt                     # create an empty file
Copy-Item notes.txt notes_backup.txt   # copy (alias: cp)
Move-Item notes_backup.txt archive\    # move or rename (alias: mv)
Remove-Item notes_backup.txt           # delete a file (alias: rm, del)
Get-Content notes.txt                  # print a file's contents (alias: cat)
```

**ℹ️ rm and Remove-Item do not use a recycle bin**

Deleting a file from the terminal is immediate and permanent — there is no undo, and (unlike deleting from Finder or File Explorer) nothing goes to the trash first. Double-check the filename, especially with a wildcard like `rm *.csv`, before pressing enter.

## Paths in python with pathlib

Never build a path by string concatenation; `pathlib` handles the `/` vs `\` difference for you, and works identically on every operating system.

In [ ]:
import os
from pathlib import Path, PurePosixPath, PureWindowsPath

print(os.name)        # 'posix' on macOS/Linux, 'nt' on Windows
print(Path.cwd())     # current working directory, in the native format

# build a path without hard-coding a separator
data_file = Path("data") / "raw" / "temperature.csv"
print(data_file)

# the same logical path rendered for each operating system
print(PurePosixPath("data/raw/temperature.csv"))     # forward slashes
print(PureWindowsPath("data/raw/temperature.csv"))   # backslashes

## Running python code

```bash
python script.py            # run a script
python -c "print(2 + 2)"    # run a one-line snippet
python                      # start an interactive interpreter (exit() to leave)
```

Keep a project's dependencies isolated in a virtual environment, rather than installing packages globally:

```bash
# macOS / Linux
python -m venv .venv
source .venv/bin/activate
```

```powershell
# Windows (PowerShell)
python -m venv .venv
.venv\Scripts\Activate.ps1
```

Once activated, `python` and `pip install` inside that terminal only affect this project. The section below covers `uv`, a faster, modern alternative to this venv-then-activate-then-pip sequence.

## uv

[`uv`](https://docs.astral.sh/uv/) is a fast, modern replacement for `pip` and `venv`: one tool that installs python itself, manages a project's virtual environment, and resolves dependencies — all noticeably faster than the tools above. Install it once, outside any project:

```bash
# macOS / Linux
curl -LsSf https://astral.sh/uv/install.sh | sh
```

```powershell
# Windows (PowerShell)
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

Inside a project, `uv` replaces the venv-then-activate-then-pip sequence above with a few commands:

```bash
uv init                    # start a new project (creates pyproject.toml)
uv add numpy pandas        # add a dependency, resolved and installed
uv run python script.py    # run inside the project's environment, no activation needed
```

`uv run` is the command to reach for by default: the first time it runs, it creates the virtual environment and installs the project's dependencies; every time after, it makes sure that environment is still in sync before running whatever follows. `uv run jupyter lab` and `uv run python script.py` always use the right, up-to-date environment, with no separate activation step to remember.

## Useful extras

- **Tab completion** — press Tab to auto-complete a file or command name; press it twice to see the options if it is ambiguous.
- **Command history** — the Up/Down arrows recall previous commands; `history` (bash/zsh) lists them.
- **Interrupting** — Ctrl+C stops a running command.
- **Getting help** — `command --help` (most unix tools) or `man command` (the full manual page); PowerShell: `Get-Help Command-Name`.
- **Chaining** — `&&` runs the next command only if the first succeeded, in both bash/zsh and PowerShell: `mkdir results && cd results`.

**🧠 Computational-thinking fundamental: the shell only knows where you told it to look**

A relative path (`data/file.csv`) is resolved against the shell's *current* directory, which is not necessarily where you think it is — the single most common source of "file not found" for anyone new to the terminal. Run `pwd`/`Get-Location` before anything else when a path fails. An absolute path (starting from `/` or `C:\`) never depends on where you happen to be, at the cost of being longer to type and less portable between machines.

<details>
<summary><b>🔍 Going deeper: WSL (Windows Subsystem for Linux)</b></summary>

WSL runs a real Linux environment alongside Windows, so every unix command on this page (and in the rest of the book) works exactly as shown, with no PowerShell translation needed. Install it with `wsl --install` in PowerShell (run as administrator), restart, then open the "Ubuntu" app it installs. Worth it if you find yourself translating between the two columns on this page more than once or twice.

</details>

<details>
<summary><b>🔍 Going deeper: shell configuration and aliases</b></summary>

`~/.zshrc` (macOS), `~/.bashrc` (Linux), or a PowerShell `$PROFILE` script runs every time you open a new terminal — a good place for a shortcut like `alias ll="ls -la"` (unix) or `function ll { Get-ChildItem -Force }` (PowerShell), so a command you type often gets shorter.

</details>

**📌 Takeaways**

- Two parallel command sets exist for everything below: unix (macOS/Linux) and PowerShell (Windows); WSL gives Windows users the unix set directly.
- Navigate with `pwd`/`cd`/`ls` (`Get-Location`/`Set-Location`/`Get-ChildItem`); manage files with `mkdir`/`cp`/`mv`/`rm` (`New-Item`/`Copy-Item`/`Move-Item`/`Remove-Item`).
- Build paths in python with `pathlib`, never by string concatenation — it handles the `/` vs `\` difference for you.
- Run a script with `python script.py`; keep a project's dependencies isolated in a virtual environment.
- `uv run <command>` gets you the same isolation as venv + pip, without a separate activation step.
- A relative path depends on the current directory; check it with `pwd`/`Get-Location` before trusting one.

## Resources

- [Software Carpentry — The Unix Shell](https://swcarpentry.github.io/shell-novice/) — hands-on introduction to shell navigation and file manipulation.
- [Microsoft Learn — PowerShell documentation](https://learn.microsoft.com/en-us/powershell/) — the official reference for every PowerShell command on this page.